## 1 - Dataset Acquisition and Initial Setup

In this first step, we are downloading the CelebDF dataset, the **3rd** version.

In [ ]:
import cv2
import glob
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [ ]:
base_path = "/Volumes/LUCIA'S HD/Celeb-DF-v3/"
folders = os.listdir(base_path)
print("Contents inside", base_path, ": ")
print('\n'.join([f"- {f}" for f in folders]))

In [ ]:
for folder in folders:
    if folder.startswith('._'):
        continue
        
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        try:
            files = [f for f in os.listdir(folder_path) if not f.startswith('._')]
            if files:
                print(f"Directory '{folder}': {len(files)} files (e.g., {files[0]})")
            else:
                print(f"Directory '{folder}': Empty")
        except Exception as e:
            print(f"Error accessing {folder}: {e}")
    else:
        print(f"File found: {folder}")

In [ ]:
synthesis_path = r"/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesis"
all_fake_videos = []

print("Looking for fake videos...")

for root, dirs, files in os.walk(synthesis_path):
    vids = [os.path.join(root, f) for f in files 
            if f.endswith(('.mp4', '.avi')) and not f.startswith('._')]
    all_fake_videos.extend(vids)

print(f"\n--- FAKE VIDEO ---")
print(f"Total: {len(all_fake_videos)}")

if all_fake_videos:
    print(f"Example path: {all_fake_videos[0]}")
    counts = {}
    for path in all_fake_videos:
        parts = path.split(os.sep)
        idx = parts.index('Celeb-synthesis')
        category = parts[idx + 1]
        counts[category] = counts.get(category, 0) + 1
    
    for cat, count in counts.items():
        print(f"- {cat}: {count} video")

In [ ]:
real_folders = ['Celeb-real', 'YouTube-real']

print(f"--- REAL VIDEO ---")

for folder in real_folders:
    folder_path = os.path.join(base_path, folder)
    print(f"\nDirectory: {folder}")
    
    if os.path.exists(folder_path):
        all_items = [f for f in os.listdir(folder_path) if not f.startswith('._')]
        
        vids = [f for f in all_items if f.endswith(('.mp4', '.avi'))]
        subdirs = [f for f in all_items if os.path.isdir(os.path.join(folder_path, f))]
        
        print(f"  - Total elements: {len(all_items)}")
        print(f"  - Total video: {len(vids)}")
        print(f"  - Subdirectory: {len(subdirs)}")
        
        if vids:
            print(f"  - Example video: {vids[:3]}")
        if subdirs:
            print(f"  - Example subdirectory: {subdirs[:3]}")
            first_sub = os.path.join(folder_path, subdirs[0])
            inner_vids = [f for f in os.listdir(first_sub) if f.endswith(('.mp4', '.avi'))]
            print(f"  - Video inside the first subdirectory ({subdirs[0]}): {len(inner_vids)}")
    else:
        print(f"  Direcotry not founded: {folder_path}")

## 2 - Pandas DataFrame

This section creates a single pandas DataFrame containing all videos from the Celeb-DF-v3 dataset.

**Columns:**
- `video`: the video filename (e.g., with `.mp4` extension).
- `full_path`: the complete file path to the video on the disk.
- `label`: 0 = real/original, 1 = fake/synthesis.
- `dataset`: the name of the dataset (e.g., "Celeb-DF-v3").
- `category`: the specific subfolder or subset category (e.g., 'Celeb-real', 'YouTube-real', or synthesis sub-categories).
- `method`: the specific deepfake generation method used, or "original" for real videos.
- `target`: ID of the target subject (the person whose face is being replaced or modified).
- `source`: ID of the source subject (the person providing the new face). For real videos, this is identical to the target.
- `sequence`: the video sequence identifier extracted from the filename, or "original" for real videos.

In [ ]:
data = []

real_folders = ['Celeb-real', 'YouTube-real']
for folder in real_folders:
    folder_path = os.path.join(base_path, folder)
    if not os.path.exists(folder_path): continue
    
    for video in os.listdir(folder_path):
        if video.endswith('.mp4') and not video.startswith('._'):
            name = video.replace('.mp4', '')
            target = name.split('_')[0] if 'id' in name else name
            
            data.append({
                "video": video,
                "label": 0,
                "dataset": "Celeb-DF-v3",
                "category": folder,
                "method": "original",
                "target": target,
                "source": target, # Reale: Source = Target
                "sequence": "original",
                "full_path": os.path.join(folder_path, video)
            })

synthesis_path = os.path.join(base_path, 'Celeb-synthesis')
if os.path.exists(synthesis_path):
    for root, dirs, files in os.walk(synthesis_path):
        for video in files:
            if video.endswith('.mp4') and not video.startswith('._'):
                name = video.replace('.mp4', '')
                parts = name.split('_')
   
                path_parts = root.split(os.sep)
                idx = path_parts.index('Celeb-synthesis')
                category = path_parts[idx + 1] if len(path_parts) > idx + 1 else "Unknown"
                method = path_parts[idx + 2] if len(path_parts) > idx + 2 else "Unknown"

                source = parts[0] if len(parts) > 0 else "Unknown"
                sequence = parts[1] if len(parts) > 1 else "Unknown"

                target = "Unknown"
                for p in parts:
                    if p.startswith('id') and p != source:
                        target = p
                        break
                
                if target == "Unknown":
                    target = source

                data.append({
                    "video": video,
                    "label": 1,
                    "dataset": "Celeb-DF-v3",
                    "category": category,
                    "method": method,
                    "target": target,
                    "source": source,
                    "sequence": sequence,
                    "full_path": os.path.join(root, video)
                })

df_celeb = pd.DataFrame(data)

print("--- CELEB-DF-V3 DATAFRAME ---")
display(df_celeb.sample(15))

### 2.1 - Sanity Check

Quick checks to ensure the dataset is loaded correctly and to understand its composition:

- **Total number of videos:** Overall count of files found on the drive.
- **Label distribution:** Balance between real (0) and fake (1) videos.
- **Method distribution:** The top AI manipulation methods used to generate the fakes.
- **Category distribution:** Breakdown by subset (e.g., Celeb-real, YouTube-real, Celeb-synthesis).
- **Identity analysis:** Counts of unique target and source IDs, including the overlap of identities present in both real and fake sets.

In [ ]:
print(f"--- CELEB-DF-V3 AUDIT ---")
print(f"Total videos found on HDD: {len(df_celeb)}")

print("\nLabel Distribution:")
print(df_celeb["label"].value_counts().rename({0: '0 (Real)', 1: '1 (Fake)'}))

print("\nAI Method Distribution (Top 10):")
print(df_celeb["method"].value_counts().head(10))

print("\nCategory Distribution:")
print(df_celeb["category"].value_counts())

print("\nIdentity Analysis (Target-based):")
unique_targets = df_celeb['target'].nunique()
print(f"Total unique Target identities: {unique_targets}")

real_targets = set(df_celeb[df_celeb['label'] == 0]['target'])
fake_targets = set(df_celeb[df_celeb['label'] == 1]['target'])
overlap = real_targets.intersection(fake_targets)

print(f"Targets present in both Real and Fake: {len(overlap)}")
print(f"Targets only in Real: {len(real_targets - fake_targets)}")
print(f"Targets only in Fake: {len(fake_targets - real_targets)}")

if df_celeb['label'].sum() > 0:
    unique_sources = df_celeb[df_celeb['label'] == 1]['source'].nunique()
    print(f"Unique Sources used for fakes: {unique_sources}")

### 2.2  - Distribution of Fake Videos per Target

Analyzes how many fake videos exist per target identity to identify if some identities dominate the fake samples

In [ ]:
fake_df = df_celeb[df_celeb["label"] == 1]
per_target = fake_df.groupby("target").size()
print("\n--- FAKE PER TARGET STATISTICS ---")
print(per_target.describe())

### 2.3 - Distribution Of Methods per Target (Check Variability):

Creates a pivot table showing how many videos of each manipulation method exist per target to verify that each identity has a representative set of manipulation methods and to detect identities with too few or missing manipulation types

In [ ]:
pivot_celeb = pd.pivot_table(
    df_celeb,
    index="target",
    columns="method",
    values="video",
    aggfunc="count",
    fill_value=0
)

print("\n--- TOP 20 TARGETS BY METHOD COVERAGE ---")
if 'original' in pivot_celeb.columns:
    display(pivot_celeb.sort_values(by='original', ascending=False).head(20))
else:
    display(pivot_celeb.sample(20))

### 2.4 - Check Label Balance

Checks the ratio of real to fake videos across the entire dataset to understand dataset imbalance

In [ ]:
print("Normalize label ditribution:")
print(df_celeb["label"].value_counts(normalize=True))

## 3 - Metadata Extraction (OpenCV)

In this section, we use **OpenCV (`cv2`)** to scan through the entire Celeb-DF-v3 dataset and extract key metadata:
- `total_frames`
- `fps`
- `duration_sec`
- `resolution`

In [ ]:
tqdm.pandas(desc="Extracting Celeb-DF-v3 metadata")

def get_video_metadata(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return pd.Series([None, None, None, None])
    
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        duration = total_frames / fps if fps > 0 else 0
        
        cap.release()
        return pd.Series([total_frames, fps, duration, f"{width}x{height}"])
    
    except Exception as e:
        return pd.Series([None, None, None, None])

print(f"Scanning {len(df_celeb)} videos. This will take a few minutes...")

df_celeb[['total_frames', 'fps', 'duration_sec', 'resolution']] = \
    df_celeb['full_path'].progress_apply(get_video_metadata)

print("\nMetadata extraction complete!")

display(df_celeb[['video', 'total_frames', 'duration_sec', 'resolution', 'method']].sample(20))

In [ ]:
print("\n--- VIDEO DURATION STATISTICS (in seconds) ---")
print(df_celeb['duration_sec'].describe())

print("\n--- VIDEO FRAMES STATISTICS ---")
print(df_celeb['total_frames'].describe())

display(df_celeb.sample(20))

## 4 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_videos/celeb_videos`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [ ]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "celeb_videos.csv")
df_celeb.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")